In [9]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ["OPENIA_API_KEY"]:
    print("API key is set")

API key is set


In [10]:
from langchain_openai import ChatOpenAI

In [11]:
llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

In [12]:
llm.invoke("what is AI? Tell me in one line")

AIMessage(content='Artificial intelligence is the field of computer science that builds machines and software capable of performing tasks that normally require human intelligence—such as learning, reasoning, and decision-making.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 426, 'prompt_tokens': 15, 'total_tokens': 441, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EeXtZfyxho49cbVCN3WCHZ7y1vz2a', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a18aec-7b05-77cb-b540-b50dedcfde70-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_t

## **RAG IMPLEMENTATION WITH YOUR OWN TEXT DATA**

#### **STEP 1 : Preparing document for your text**

In [16]:
from langchain_core.documents import Document

## Your text data
my_text = """Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is also a field of research spanning computer science, engineering, and mathematics that develops and studies methods and software enabling machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]
Prominent applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).
The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers use techniques including state space search and mathematical optimisation, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind, and Meta, aim to create artificial general intelligence (AGI)—AI that can complete nearly any cognitive task at least as well as a human.[3] 
"""

docs = [Document(page_content=my_text, metadata={"source":"ABC","documentID":"Doc1"})]

#### **STEP 2 : Splitting the document into Chunks**

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is also a field of research spanning computer science, engineering, and mathematics that develops and studies methods and software enabling machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='that maximise their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='Prominent applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images,

#### **STEP 3 : Creating Embeddings for the Chunks**

In [23]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings()

#### **STEP 4 : Create and store embeddings in vector store**

In [24]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

#### **STEP 5 : Similarity Seach**

In [ ]:
vectorstore.similarity_search("What is AI", k=3)

[Document(metadata={'documentID': 'Doc1', 'source': 'ABC'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is also a field of research spanning computer science, engineering, and mathematics that develops and studies methods and software enabling machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='Prominent applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='philosophy, neuroscience, and other fi

#### **TALK LMM**

In [26]:
context = vectorstore.similarity_search("What is AI", k=3)

In [29]:
response = llm.invoke(f"What is AI? You can answer using the following context: {context}")
print(response.content)

- Definition: AI is the capability of computational systems to perform tasks typically associated with human intelligence—such as learning, reasoning, problem-solving, perception, and decision-making. It is also a field of research across computer science, engineering, and mathematics that develops methods and software enabling machines to perceive their environment, learn, and take actions to achieve defined goals.

- Applications: AI is used in advanced web search, chatbots, virtual assistants, autonomous vehicles, strategy games analysis (e.g., chess and Go), and content generation (text, images, audio, videos), among others.

- AGI note: Some groups aim to create artificial general intelligence (AGI), AI that can complete nearly any cognitive task at least as well as a human.
